# 01 — Data Exploration

**Objetivo:** Verificar la calidad, cobertura y estructura de los datos descargados.

Preguntas que responde este notebook:
- ¿Cuántas carreras y pilotos hay en el dataset?
- ¿Cuál es la cobertura de datos de qualifying?
- ¿Hay valores faltantes importantes?
- ¿Cómo se distribuyen los DNFs?
- ¿Está el dataset correctamente ordenado cronológicamente?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR, HISTORY
from src.data.load_results import load_results
from src.data.load_qualifying import load_qualifying

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
seasons = list(range(HISTORY['start_year'], HISTORY['end_year'] + 1))
results = load_results(seasons)
qualifying = load_qualifying(seasons)

print(f"Race results: {len(results):,} rows")
print(f"Qualifying:   {len(qualifying):,} rows")
print(f"Seasons:      {results['season'].min()} – {results['season'].max()}")
print(f"Races:        {results.groupby(['season','round']).ngroups}")
print(f"Drivers:      {results['driver_id'].nunique()}")
print(f"Constructors: {results['constructor_id'].nunique()}")

In [ ]:
# Qualifying coverage by season
qual_coverage = (
    qualifying.groupby('season')['qualifying_position']
    .count()
    .reset_index(name='q_rows')
)
race_rows = results.groupby('season').size().reset_index(name='race_rows')
coverage = race_rows.merge(qual_coverage, on='season', how='left')
coverage['coverage_pct'] = (coverage['q_rows'] / coverage['race_rows'] * 100).round(1)

fig, ax = plt.subplots()
ax.bar(coverage['season'], coverage['coverage_pct'], color='steelblue')
ax.axhline(100, color='green', linestyle='--', label='100%')
ax.set_xlabel('Season')
ax.set_ylabel('Qualifying coverage (%)')
ax.set_title('Qualifying data coverage by season')
ax.legend()
plt.tight_layout()
plt.show()
print(coverage.to_string(index=False))

In [ ]:
# Missing values in results
print("=== Missing values (race results) ===")
print(results.isnull().sum().to_string())
print()
print("=== Missing values (qualifying) ===")
print(qualifying.isnull().sum().to_string())

In [ ]:
# DNF rate by season
dnf_by_season = results.groupby('season')['dnf'].mean() * 100
dnf_by_season.plot(kind='bar', title='DNF rate by season (%)', ylabel='DNF %')
plt.tight_layout()
plt.show()

In [ ]:
# Races per season
races_per_season = results.groupby('season')['round'].max()
print("Races per season:")
print(races_per_season.to_string())

In [ ]:
# Qualifying session reached distribution
qual_session = qualifying['qualifying_session_reached'].value_counts()
print("Qualifying session reached distribution:")
print(qual_session)
qual_session.plot(kind='bar', title='Qualifying session reached (Q1/Q2/Q3)')
plt.tight_layout()
plt.show()

In [ ]:
# Verify chronological ordering
assert results['race_date'].is_monotonic_increasing, "Results are NOT chronologically ordered!"
print("✅ Results are chronologically ordered.")
print(f"First race: {results.iloc[0]['race_date']} — {results.iloc[0]['race_name']}")
print(f"Last race:  {results.iloc[-1]['race_date']} — {results.iloc[-1]['race_name']}")